In [0]:
# output= dbutils.notebook.run("/Workspace/Users/fizulhaq.1987@gmail.com/DemoRepo/My_Folder/bakehouse/environmental variables",60)
# table_column= json.loads(output)


In [0]:
%run "./environmental_variables"

In [0]:
'''what: Data ingestion from source to bronze
   how: Read the source raw tables and store it in bronze as Delta files
   why: Bronze is the raw layer and it has no transformation and No need for table Metadata 
'''
from pyspark.sql import functions as F

def load_to_bronze(table,column):
    print(f"source table load started for {table}")
    path=f"{config['bronze_path']}/{table}"
    val = config['source_tables']
    source_table=val[table]
    if spark.catalog.tableExists(source_table):
        df= spark.read.table(source_table).select(*column).withColumn("ingestion_date",F.lit(ingestion_date))
        df.write.format("delta").option("mergeSchema","true").mode("overwrite").partitionBy('ingestion_date').save(path)
        print(f"source table load completed for {table}")
    else:
        print(f"source table {source_table} does not exists")


for tab,col in table_col_zip:
    load_to_bronze(tab,col)


In [0]:
'''Data Quality Function used to validate the rules needs to set on certain columns
'''
from pyspark.sql import functions as F
def apply_dq_checks(df, dq_rules):
    error_cols = []
    for col_name, rules in dq_rules.items():

        for rule, message in rules.items():

            if rule == "not_null":
                error_col = F.when(F.col(col_name).isNull(), F.lit(message))
                
            elif rule == "positive":
                error_col = F.when(F.col(col_name) <= 0, F.lit(message))

            elif rule == "valid_date":
                error_col = F.when(F.col(col_name).isNull()|
                                   ((F.to_date(F.col(col_name)) < F.to_date(F.lit("2020-01-01")))|
                                    (F.to_date(F.col(col_name)) > F.current_date())), F.lit(message))

            else:
                continue

            error_cols.append(error_col)

    # Combine all error messages into an array
    df = df.withColumn(
        "dq_errors",
        F.array(*[c for c in error_cols])
        )

    # Filter out nulls inside the array
    df = df.withColumn(
        "dq_errors",
         F.expr("filter(dq_errors, x -> x is not null)")
        )

    # Add pass/fail flag
    df = df.withColumn(
        "dq_status",
        F.when(F.size("dq_errors") > 0, "FAIL").otherwise("PASS")
        )

    return df

In [0]:
''' DQC data quality checks handled on each of source master and transaction table and save those bad records into the
    delta table for further anayalysis
'''

def dcq_bad_rawdata_load(dq_rules,table_name,bad_data_tblname):
    path=f"{config['bronze_path']}/{table_name}"
    df = spark.read.format("delta").load(path)
    df_dqc = apply_dq_checks(df,dq_rules)
    bad_data_count = df_dqc.filter(F.col('dq_status')=='FAIL').limit(1).count()
    if bad_data_count > 0:
        df_dqc.filter(F.col('dq_status')=='FAIL').write.mode("overwrite").format("delta").partitionBy('ingestion_date').saveAsTable(f"inceptezcatalog.bakehouse.{bad_data_tblname}")
        print(f"{table_name} bad data count:- {bad_data_count}")
    else:
        print(f"No bad data in {table_name} master data")

dcq_bad_rawdata_load(dq_rules_customer_source,'sales_customers','src_cust_bad_data')
dcq_bad_rawdata_load(dq_rules_transaction_source,'sales_transactions','src_tran_bad_data')
dcq_bad_rawdata_load(dq_rules_franchise_source,'sales_franchise','src_franchise_bad_data')
dcq_bad_rawdata_load(dq_rules_supplier_source,'sales_suppliers','src_supplier_bad_data')